# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:**

Change over time calls for a line chart. I index each continent to 1970 = 100 so the lines show growth relative to 1970 size. Africa gets the accent colour, the other continents are grey, and direct end labels replace the legend.

In [2]:
import plotly.graph_objects as go

g = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
g['Index'] = g['Population'] / g.groupby('Continent')['Population'].transform('first') * 100

fastest = g[g.Year == 2022].sort_values('Index').iloc[-1]
colors = {c: '#BBBBBB' for c in g.Continent.unique()}
colors[fastest.Continent] = '#E69F00'   # accent (CVD-safe orange)

fig = px.line(g, x='Year', y='Index', color='Continent', color_discrete_map=colors,
              template='plotly_white',
              title=f"{fastest.Continent}'s population has grown ~{fastest.Index/100:.1f}x since 1970, "
                    f"far faster than any other continent")
fig.update_traces(line_width=2)
fig.update_traces(line_width=4, selector=dict(name=fastest.Continent))

# direct end labels
end = g[g.Year == 2022].sort_values('Index').reset_index(drop=True)
ypos = end['Index'].tolist()
for k in range(1, len(ypos)):
    ypos[k] = max(ypos[k], ypos[k-1] + 15)      # minimum gap between labels
for (_, r), y in zip(end.iterrows(), ypos):
    fig.add_annotation(x=2022, y=y, text=f" {r.Continent} ({r.Index:.0f})", showarrow=False,
                       xanchor='left',
                       font=dict(size=12, color=colors[r.Continent] if r.Continent == fastest.Continent else '#666'))
fig.add_hline(y=100, line_dash='dot', line_color='#999')
fig.update_layout(showlegend=False, margin=dict(r=140), yaxis_title='Population (1970 = 100)',
                  xaxis_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:**

This is a ranking of 15 items, so a sorted bar chart is best. Horizontal bars leave room for the country names. Africa is highlighted in orange and everything else is grey, which makes the “how many are in Africa” count visible at a glance.

In [3]:
d = df.assign(Added=df['2022 Population'] - df['1970 Population'])
top = d.nlargest(15, 'Added').sort_values('Added')          # ascending -> largest on top in horizontal bar
n_africa = (top.Continent == 'Africa').sum()
top['Is Africa'] = np.where(top.Continent == 'Africa', 'Africa', 'Other')

fig = px.bar(top, x='Added', y='Country', color='Is Africa', orientation='h',
             color_discrete_map={'Africa': '#E69F00', 'Other': '#BBBBBB'},
             category_orders={'Country': top.Country.tolist()},
             text=(top['Added'] / 1e6).round(0).astype(int).astype(str) + 'M',
             template='plotly_white',
             title=f'{n_africa} of the 15 countries that added the most people since 1970 are in Africa')
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title=None,
                  xaxis_visible=False, margin=dict(r=60))
fig.update_yaxes(showgrid=False)
fig.update_yaxes(autorange='reversed')
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:**

Two numeric variables mean a scatter plot. Both axes are log-scaled because area and population span several orders of magnitude, and colour encodes continent (Okabe-Ito palette, which is CVD-safe). The strong log-scale relationship (r = 0.87) holds on every continent, as the colours show; within-continent correlations range from 0.74 to 0.94. Labels mark the real exceptions found from the largest residuals of the log-log trend line, plus China and India as the largest countries.

In [4]:
cvd = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#009E73', 'North America': '#CC79A7',
       'Oceania': '#56B4E9', 'South America': '#999999'}

d = df.copy()
lx, ly = np.log10(d['Area']), np.log10(d['2022 Population'])
slope, intercept = np.polyfit(lx, ly, 1)
d['resid'] = ly - (slope * lx + intercept)
outliers = d.reindex(d.resid.abs().sort_values(ascending=False).index).head(5).Country.tolist()
outliers += ['China', 'India']          # largest countries, for context

r = np.corrcoef(lx, ly)[0, 1]
title = (f'Bigger countries have more people on every continent (r = {r:.2f}, log scales); '
         'Greenland and the Falklands are far emptier than their size predicts, '
         'Macau and Singapore far fuller')

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent', color_discrete_map=cvd,
                 hover_name='Country', log_x=True, log_y=True, opacity=0.8,
                 template='plotly_white', title=title)

# trend line (log-log)
xs = np.linspace(lx.min(), lx.max(), 50)
fig.add_scatter(x=10**xs, y=10**(slope * xs + intercept), mode='lines',
                line=dict(color='#444', dash='dot'), showlegend=False, hoverinfo='skip')

# label outliers
for _, r in d[d.Country.isin(set(outliers))].iterrows():
    fig.add_annotation(x=np.log10(r.Area), y=np.log10(r['2022 Population']), text=r.Country,
                       showarrow=True, arrowhead=0, ax=25, ay=-20, font=dict(size=11))

fig.update_traces(marker=dict(size=9, line=dict(width=0)), selector=dict(mode='markers'))
fig.update_layout(xaxis_title='Land area (km², log scale)', yaxis_title='Population 2022 (log scale)',
                  legend_title_text=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:**

The question asks for the spread within and across groups, which is what a box plot shows (median, quartiles, outliers) for each continent. Density is extremely right-skewed, so the y-axis is log-scaled. Outliers are shown as points and hoverable by country name. Continents are sorted by median, and the continent with the highest median gets the accent. North America’s high median is driven largely by small Caribbean territories.

In [5]:
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
top_cont = order.index[0]
colors = {c: '#BBBBBB' for c in order.index}
colors[top_cont] = '#0072B2'

fig = px.box(df, x='Continent', y='Density', color='Continent', color_discrete_map=colors,
             points='outliers', hover_name='Country', log_y=True,
             category_orders={'Continent': order.index.tolist()},
             template='plotly_white',
             title=f'{top_cont} is the most crowded continent at the median, but every continent '
                   'has huge internal spread')
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title='People per km² (log scale)')

# label the most extreme countries
for _, r in df.nlargest(3, 'Density').iterrows():
    fig.add_annotation(x=r.Continent, y=np.log10(r.Density), text=r.Country, showarrow=True,
                       arrowhead=0, ax=40, ay=0, font=dict(size=11))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:**

The question asks where in the world countries grew or shrank, so it needs a map. A diverging scale centred on 0 separates growth from decline, using orange (shrank) and blue (grew), which is CVD-safe, unlike red/green. The colour range is clipped at -50% and +150%; only Qatar (+317%) and the UAE (+188%) exceed the +150% clip, and hover still shows the true value. Many shrinking territories, especially Pacific and Caribbean islands, are too small to see on the map.

In [6]:
d = df.copy()
d['Change'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100
d['Change (clipped)'] = d['Change'].clip(-50, 150)
n_shrunk = (d['Change'] < 0).sum()

fig = px.choropleth(d, locations='CCA3', locationmode='ISO-3', color='Change (clipped)',
                    hover_name='Country', hover_data={'Change': ':.1f', 'Change (clipped)': False, 'CCA3': False},
                    color_continuous_scale=[(0, '#D55E00'), (0.25, '#F5F5F5'), (1, '#0072B2')],
                    range_color=(-50, 150),
                    template='plotly_white',
                    title=f'Most countries grew since 2000, but {n_shrunk} shrank, led by Eastern Europe and the Baltics')
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(coloraxis_colorbar=dict(title='% change<br>2000–2022', len=0.6,
                                          tickvals=[-50, 0, 50, 100, 150],
                                          ticktext=['≤-50', '0', '50', '100', '≥150']),
                  margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:**

Part-to-whole data with a two-level hierarchy (continent, then country) is what a treemap is for. Tile area is population, and colour is the continent so it encodes meaning, with a CVD-safe palette. Continents are the outer groups and countries the inner tiles.

In [7]:
cvd = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#009E73', 'North America': '#CC79A7',
       'Oceania': '#56B4E9', 'South America': '#999999', '(?)': '#FFFFFF'}

asia_share = df.loc[df.Continent == 'Asia', '2022 Population'].sum() / df['2022 Population'].sum()
ci = df.loc[df.Country.isin(['China', 'India']), '2022 Population'].sum() / df['2022 Population'].sum()

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'],
                 values='2022 Population', color='Continent', color_discrete_map=cvd,
                 template='plotly_white',
                 title=f'Asia holds {asia_share:.0%} of the world; China and India alone are {ci:.0%}, twice Africa’s share')
fig.update_traces(textinfo='label+percent root', root_color='white',
                  marker=dict(line=dict(width=1, color='white')))
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:**

Shares of a whole over time call for a 100% stacked bar chart, since the segments always add up to 100% and the shift between groups is easy to read. Years are treated as categories because the gaps are uneven (10 years, then 5, then 2). Africa is the accent, Asia and Europe get muted colours, and the rest is grey. Group names are direct labels at the right edge, and percentages are inside the segments.

In [8]:
d = df_long.copy()
d['Group'] = np.where(d.Continent.isin(['Africa', 'Asia', 'Europe']), d.Continent, 'Rest of the world')
s = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
s['Share'] = s['Population'] / s.groupby('Year')['Population'].transform('sum') * 100
s['Year'] = s['Year'].astype(str)

order = ['Africa', 'Asia', 'Europe', 'Rest of the world']
cmap = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#56B4E9', 'Rest of the world': '#CCCCCC'}

af = s[s.Group == 'Africa'].set_index('Year')['Share']
eu = s[s.Group == 'Europe'].set_index('Year')['Share']

fig = px.bar(s, x='Year', y='Share', color='Group', color_discrete_map=cmap,
             category_orders={'Group': order}, text=s['Share'].round(0).astype(int).astype(str) + '%',
             template='plotly_white',
             title=f"Africa's share rose from {af['1970']:.0f}% to {af['2022']:.0f}% since 1970, "
                   f"while Europe's fell from {eu['1970']:.0f}% to {eu['2022']:.0f}%")
fig.update_traces(textposition='inside', textfont_size=11)
fig.update_layout(barmode='stack', showlegend=False, margin=dict(r=150),
                  xaxis_title=None, yaxis_title=None, yaxis_visible=False)

last = s[s.Year == '2022'].set_index('Group').loc[order]
mid = last.Share.cumsum() - last.Share / 2
for grp, y in mid.items():
    fig.add_annotation(x=7.5, y=y, text=grp, showarrow=False, xanchor='left',
                       font=dict(size=12, color='#444'))
fig.update_xaxes(showgrid=False)
fig.show()